In [4]:
using MLDatasets
using Statistics
using Random
using LinearAlgebra
using Flux: logitcrossentropy, onehotbatch, onecold

BLAS.set_num_threads(1)
include("clothesolver.jl")

train_data = MLDatasets.FashionMNIST(split=:train)
test_data = MLDatasets.FashionMNIST(split=:test)

X_train = Float32.(reshape(train_data.features, 28, 28, 1, :))
X_test = Float32.(reshape(test_data.features, 28, 28, 1, :))

Y_train_raw = train_data.targets
Y_test_raw = test_data.targets

Y_train = Float32.(onehotbatch(Y_train_raw, 0:9))
Y_test = Float32.(onehotbatch(Y_test_raw, 0:9))

println("Sizes: ", size(X_train))

Sizes: (28, 28, 1, 60000)


In [5]:
my_net_def = Chain(
  Conv((3, 3), 1 => 6, bias=false),
  MaxPool((2, 2)),
  Conv((3, 3), 6 => 16, bias=false),
  MaxPool((2, 2)),
  Flatten(),
  Dense(784 => 84, relu),
  Dropout(0.4),
  Dense(84 => 10)
)

function allocate(net_def, X_tr)
    model = build_model(net_def, (28, 28, 1); batch_size=10)
    idxs = collect(1:size(X_tr, 4))
    return model, idxs
end

function test(model::CompiledModel, X, Y_raw)
    correct = 0
    N = size(X, 4)
    bs = model.batch_size
    input_node = model.input
    final_layer = model.chain.layers[end]

    x_flat = parent(input_node.data)
    len_x = 28 * 28 * 1

    for i in 1:bs:N
        b_end = min(i + bs - 1, N)
        actual_b = b_end - i + 1

        if actual_b < bs
            fill!(input_node.data, 0f0)
        end

        copyto!(x_flat, 1, X, (i - 1) * len_x + 1, actual_b * len_x)
        forward_test!(model.chain, input_node)

        @inbounds for b in 1:actual_b
            max_v = -Inf32
            max_idx = 1
            for c in 1:10
                v = final_layer.out.data[c, b]
                if v > max_v
                    max_v = v
                    max_idx = c
                end
            end
            pred_class = max_idx - 1
            correct += (pred_class == Y_raw[i + b - 1])
        end
    end

    acc = round(correct / N * 100, digits=2)
    return acc
end

function train!(model::CompiledModel, X, Y, indices; η::Float32)
    bs = model.batch_size
    input_node = model.input
    target_node = model.target
    loss_fn = model.loss
    N = size(X, 4)
    total_loss = 0.0
    n_batches = 0

    x_flat = parent(input_node.data)
    y_flat = parent(target_node.data)
    len_x = 784
    len_y = 10

    Random.shuffle!(indices)

    for b_start in 1:bs:N
        b_end = min(b_start + bs - 1, N)
        b_end - b_start + 1 < bs && break
        
        actual_bs = b_end - b_start + 1

        zero_w_grad!(model.pool)
        zero_a_grad!(model.pool)

        for i in 1:actual_bs
            idx = indices[b_start + i - 1]
            copyto!(x_flat, (i - 1) * len_x + 1, X, (idx - 1) * len_x + 1, len_x)
            copyto!(y_flat, (i - 1) * len_y + 1, Y, (idx - 1) * len_y + 1, len_y)
        end

        preds = forward_train!(model.chain, input_node)
        batch_loss = primal!(loss_fn, preds, target_node)

        loss_fn.out.grad[1] = 1.0f0
        adjoint!(loss_fn, preds, target_node)
        backward!(model.chain, input_node)

        optimize!(model.pool, η)
        total_loss += batch_loss
        n_batches += 1
    end
    return total_loss / n_batches
end

function run_test(model, X_tr, Y_tr, X_te, Y_te, idxs)
    println("\n[x] Training...")

    @time for ep in 1:3
        t = @timed train!(model, X_tr, Y_tr, idxs; η=0.01f0)
        L = t.value
        acc = test(model, X_te, Y_te)
        n_alloc = Base.gc_alloc_count(t.gcstats)
        println("[+] Epoch $ep | Loss: $(round(L, digits=4)) | Acc: $acc% | " *
                "Time: $(round(t.time, digits=2))s | " *
                "Allocations: $n_alloc ($(Base.format_bytes(t.bytes)))")
    end
end

println("Pre-allocation:")
@time my_model, indices = allocate(my_net_def, X_train)

run_test(my_model, X_train, Y_train, X_test, Y_test_raw, indices)

Pre-allocation:
  0.815092 seconds (1.11 M allocations: 60.599 MiB, 4.95% gc time, 99.69% compilation time)

[x] Training...
[+] Epoch 1 | Loss: 0.594 | Acc: 85.14% | Time: 5.71s | Allocations: 0 (0 bytes)
[+] Epoch 2 | Loss: 0.4283 | Acc: 85.87% | Time: 5.62s | Allocations: 0 (0 bytes)
[+] Epoch 3 | Loss: 0.3877 | Acc: 87.15% | Time: 5.6s | Allocations: 0 (0 bytes)
 18.419734 seconds (361 allocations: 22.344 KiB)


In [ ]:
using Flux

flux_net_def = Flux.Chain(
  Flux.Conv((3, 3), 1 => 6, pad=1, bias=false),
  Flux.MaxPool((2, 2)),
  Flux.Conv((3, 3), 6 => 16, pad=1, bias=false),
  Flux.MaxPool((2, 2)),
  Flux.flatten,
  Flux.Dense(784 => 84, Flux.relu),
  Flux.Dropout(0.4),
  Flux.Dense(84 => 10)
)

function allocate_flux(net_def)
    model = deepcopy(net_def)
    opt_state = Flux.setup(Descent(0.01), model) 
    return model, opt_state
end

function test_flux(model, X, Y_raw)
    Flux.testmode!(model)
    preds = model(X)
    pred_classes = onecold(preds, 0:9)
    acc = round(mean(pred_classes .== Y_raw) * 100, digits=2)
    return acc
end

function train_flux!(model, opt_state, X, Y; batch_size)
    Flux.trainmode!(model)
    N = size(X, 4)
    data_loader = Flux.DataLoader((X, Y), batchsize=batch_size, shuffle=true)
    total_loss = 0.0

    for (x_batch, y_batch) in data_loader
        val, grads = Flux.withgradient(model) do m
            preds = m(x_batch)
            logitcrossentropy(preds, y_batch)
        end
        Flux.update!(opt_state, model, grads[1])
        total_loss += val * size(x_batch, 4)
    end
    return total_loss / N
end

function run_test_flux(model, opt_state, X_tr, Y_tr, X_te, Y_te)
    println("\n[x] Training...")

    @time for ep in 1:3
        t = @timed train_flux!(model, opt_state, X_tr, Y_tr, batch_size=10)
        L = t.value
        acc = test_flux(model, X_te, Y_te)
        n_alloc = Base.gc_alloc_count(t.gcstats)
        println("[+] Epoch $ep | Loss: $(round(L, digits=4)) | Acc: $acc% | " *
                "Time: $(round(t.time, digits=2))s | " *
                "Allocations: $n_alloc ($(Base.format_bytes(t.bytes)))")
    end
end

println("Pre-allocation:")
@time my_flux_model, flux_opt_state = allocate_flux(flux_net_def)

run_test_flux(my_flux_model, flux_opt_state, X_train, Y_train, X_test, Y_test_raw)

In [ ]:
using PythonCall
using CondaPkg

torch = pyimport("torch")
np = pyimport("numpy")
nn = torch.nn
optim = torch.optim

torch.set_num_threads(1)

model = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=3, padding=1, bias=false),
    nn.MaxPool2d(2),
    nn.Conv2d(6, 16, kernel_size=3, padding=1, bias=false),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(784, 84),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(84, 10)
)

function to_torch(X::AbstractArray, Y::AbstractVector)
    np = pyimport("numpy")
    torch = pyimport("torch")

    X_np = np.asarray(X)
    X_np = np.transpose(X_np, (3, 2, 0, 1))
    X_np = np.ascontiguousarray(X_np, dtype=np.float32)
    X_t = torch.from_numpy(X_np)

    Y_np = np.asarray(Y, dtype=np.int64)
    Y_np = np.ascontiguousarray(Y_np)
    Y_t = torch.from_numpy(Y_np).to(dtype=torch.long)

    return X_t, Y_t
end

function allocate_pytorch()
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    return model, optimizer, criterion
end

function test_pytorch(model, X, Y)
    model.eval()
    
    pywith(torch.no_grad()) do _
        preds = model(X)
        pred_classes = torch.argmax(preds, dim=1)
        
        p_jl = pyconvert(Vector{Int}, pred_classes)
        y_jl = pyconvert(Vector{Int}, Y)
        
        acc = round(Float64(mean(p_jl .== y_jl)) * 100, digits=2)
        return acc
    end
end

function train_pytorch!(model, optimizer, criterion, X, Y; batch_size=10)
    model.train()
    n_samples = pyconvert(Int, X.shape[0])
    total_loss = 0.0
    n_batches = 0

    for start_idx in 1:batch_size:n_samples
        end_idx = min(start_idx + batch_size - 1, n_samples)
        
        batch_X = X[pybuiltins.slice(start_idx-1, end_idx)]
        batch_Y = Y[pybuiltins.slice(start_idx-1, end_idx)]

        optimizer.zero_grad()
        preds = model(batch_X)
        loss = criterion(preds, batch_Y)
        loss.backward()
        optimizer.step()

        total_loss += pyconvert(Float64, loss.item())
        n_batches += 1
    end

    return total_loss / n_batches
end

function run_test_pytorch(model, optimizer, criterion, X_tr, Y_tr, X_te, Y_te)
    @time for ep in 1:3
        L = train_pytorch!(model, optimizer, criterion, X_tr, Y_tr; batch_size=10)
        acc = test_pytorch(model, X_te, Y_te)
        println("[+] Epoch $ep | Loss: $(round(L, digits=4)) | Acc: $acc%")
    end
end

X_train_torch, Y_train_torch = to_torch(X_train, Y_train_raw)
X_test_torch,  Y_test_torch  = to_torch(X_test,  Y_test_raw)

println("Pre-allocation:")
@time my_py_model, my_py_opt, my_py_crit = allocate_pytorch()

run_test_pytorch(my_py_model, my_py_opt, my_py_crit, X_train_torch, Y_train_torch, X_test_torch, Y_test_torch)

In [ ]:
using Plots

# https://github.com/zalandoresearch/fashion-mnist
const LABELS = Dict(
    0 => "T-shirt/top",
    1 => "Trouser",
    2 => "Pullover",
    3 => "Dress",
    4 => "Coat",
    5 => "Sandal",
    6 => "Shirt",
    7 => "Sneaker",
    8 => "Bag",
    9 => "Ankle boot"
)

num_images = 5
indices = rand(1:size(X_test, 4), num_images)

final_layer = my_model.chain.layers[end]
input_node = my_model.input

Flux.testmode!(my_flux_model)
my_py_model.eval()

plot_array = []

for idx in indices
    true_label_idx = Y_test_raw[idx]

    fill!(input_node.data, 0f0)
    copyto!(@view(input_node.data[:, :, :, 1:1]), @view(X_test[:, :, :, idx:idx]))
    forward_test!(my_model.chain, input_node)
    my_pred_idx = argmax(@view(final_layer.out.data[:, 1])) - 1
    flux_out = my_flux_model(X_test[:, :, :, idx:idx])
    flux_pred_idx = Flux.onecold(flux_out, 0:9)[1]
    pt_pred_idx = pywith(torch.no_grad()) do _
        pt_batch = X_test_torch[pybuiltins.slice(idx-1, idx)]
        pt_out = my_py_model(pt_batch)
        return pyconvert(Int, torch.argmax(pt_out, dim=1).item())
    end

    println(" ==== Image $idx ==== ")
    println("  Real:    ", LABELS[true_label_idx])
    println("  My:      ", LABELS[my_pred_idx])
    println("  Flux:    ", LABELS[flux_pred_idx])
    println("  PyTorch: ", LABELS[pt_pred_idx])

    img_data   = X_test[:, :, 1, idx]
    title_text = "Real: $(LABELS[true_label_idx])\nMy: $(LABELS[my_pred_idx])\nFlux: $(LABELS[flux_pred_idx])\nPT: $(LABELS[pt_pred_idx])"

    p = heatmap(
        img_data',
        yflip = true,
        color = :grays,
        legend = false,
        aspect_ratio = :equal,
        axis = false,
        title = title_text,
        titlefontsize = 8
    )
    push!(plot_array, p)
end

plot(plot_array..., layout=(1, num_images), size=(1200, 350), margin=5Plots.mm)